# 02 · Audit a recipe collection

Any set of recipes that claims to be a cuisine can be scored against the real CulinaryDB cuisine, whether people or an AI wrote it. The scoring uses the same metrics as the project's D2 audit. This notebook:

1. takes the **160 Korean recipes that GPT-6-Luna wrote** for LLMCuisine-22 (deliverable D1, `data/release/LLMCuisine-22/recipes_full.jsonl`);
2. aliases every raw ingredient line to a CulinaryDB entity with the CulinaryDB protocol (`World.alias`);
3. builds a cuisine with `World.cuisine_from_lines` and scores it with `World.scorecard`;
4. scores a genuine CulinaryDB sample of the same size in the same way, to show what sampling noise alone looks like;
5. runs the same audit from the command line (`flavoraudit audit`).

> This is a worked example of the API on one model and one region. It is **not** the D2 analysis. The D2 analysis covers every model and region with equal-N reference distributions and pre-registered tests with FDR control (`analysis/audit/`). Do not draw conclusions about the model from this notebook.

**Data and licences.**
- LLMCuisine-22 is CC BY-NC-SA 4.0, because it is aliased to CulinaryDB. The model output terms are in its datasheet.
- CulinaryDB and FlavorDB (CC BY-NC-SA 3.0) and the Ahn et al. data (CC BY 4.0) are read from `data/raw`.

**Run time: about 2 minutes.** Most of it is spent building the reference statistics of the 22 real cuisines, once in the notebook and once in the command-line run.

In [1]:
import hashlib
import os
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

import flavoraudit as fa
from flavoraudit.audit import real_subsample

# The repository root: the first folder upwards whose data/raw holds the source databases.
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data" / "raw" / "culinarydb").is_dir())
DATA = REPO / "data" / "raw"
RELEASE = REPO / "data" / "release" / "LLMCuisine-22"
MODEL, REGION = "openai-gpt-6-luna", "Korea"
print(f"flavoraudit {fa.__version__}")

flavoraudit 0.1.0.dev0


In [2]:
llm = pd.read_json(RELEASE / "recipes_full.jsonl", lines=True)
digest = hashlib.sha256((RELEASE / "recipes_full.jsonl").read_bytes()).hexdigest()
batch = llm[(llm["model"] == MODEL) & (llm["cuisine"] == REGION)].reset_index(drop=True)
print(f"LLMCuisine-22 recipes_full.jsonl (sha256 {digest[:12]}…): {len(llm):,} recipes")
print(f"{MODEL} × {REGION}: {len(batch)} recipes, by prompt condition {batch['condition'].value_counts().to_dict()}")
example = batch.iloc[0]
print(f"\nExample: {example['title']}")
print("\n".join(f"  {line}" for line in example["ingredients"]))

LLMCuisine-22 recipes_full.jsonl (sha256 4f96f9c00ae8…): 11,656 recipes
openai-gpt-6-luna × Korea: 160 recipes, by prompt condition {'plain': 80, 'persona': 80}

Example: Bibimbap
  2 cups short-grain rice
  200 g beef, thinly sliced
  1 tablespoon soy sauce
  1 teaspoon sesame oil
  1 teaspoon sugar
  1 carrot, julienned
  1 zucchini, julienned
  100 g spinach
  100 g bean sprouts
  4 eggs
  2 tablespoons gochujang
  2 tablespoons vegetable oil
  1 tablespoon sesame seeds
  1 teaspoon salt


## Aliasing raw ingredient lines

`World.alias` maps a raw line to a CulinaryDB entity id with the CulinaryDB protocol. In held-out tests it agrees with CulinaryDB's expert mapping on 97.9% of unique lines and 99.0% of line occurrences (`analysis/aliasing/evaluate_aliaser.py`). The match types are:
- `full`: a dictionary name or synonym matched completely;
- `partial`: the best partial match of a multi-part synonym, the kind of match CulinaryDB's curators accepted;
- `none`: unmatched, so the line is dropped, as in CulinaryDB.

The aliaser is built on first use (a few seconds).

In [3]:
world = fa.load_world(DATA)
names = world.profiles.names  # CulinaryDB entity id -> ingredient name
matches = [world.aliaser.alias(line) for line in example["ingredients"]]
pd.DataFrame(
    {
        "line": example["ingredients"],
        "entity_id": pd.array([m.entity_id for m in matches], dtype="Int64"),
        "CulinaryDB ingredient": [names.get(m.entity_id, "-") for m in matches],
        "match": [m.how for m in matches],
    }
)

,line,entity_id,CulinaryDB ingredient,match
0,2 cups short-grain rice,55,Rice,full
1,"200 g beef, thinly sliced",270,Beef,full
2,1 tablespoon soy sauce,291,Soybean Sauce,full
3,1 teaspoon sesame oil,299,Sesame,full
4,1 teaspoon sugar,781,Sugar,full
5,"1 carrot, julienned",368,Carrot,full
6,"1 zucchini, julienned",969,Zucchini,full
7,100 g spinach,396,Spinach,full
8,100 g bean sprouts,284,Beans,full
9,4 eggs,0,Egg,full


In [4]:
lines = batch[["recipe_id", "ingredients"]].explode("ingredients").rename(columns={"ingredients": "line"})
lines["match"] = [world.aliaser.alias(line).how for line in lines["line"]]
print(f"{len(lines):,} lines; share by match type: {lines['match'].value_counts(normalize=True).round(3).to_dict()}")
print("Unmatched examples:", lines.loc[lines["match"] == "none", "line"].head(6).tolist())

# The released D1 aliasing was made with the same production aliaser, so the library should reproduce it exactly.
released = pd.read_csv(RELEASE / "04_Recipe-Ingredients_Aliases.csv", dtype={"Entity ID": "Int64"})
released = released[released["Recipe ID"].isin(set(batch["recipe_id"]))]
ours = pd.array([world.alias(line) for line in released["Original Ingredient Name"]], dtype="Int64")
agree = np.mean(ours.fillna(-1).to_numpy() == released["Entity ID"].fillna(-1).to_numpy())
print(f"World.alias reproduces the released LLMCuisine-22 aliasing for {agree:.1%} of {len(released):,} lines")

1,653 lines; share by match type: {'full': 0.91, 'none': 0.071, 'partial': 0.019}
Unmatched examples: ['2 tablespoons gochujang', '1 tablespoon gochugaru', '1 tablespoon gochujang', '2 tablespoons gochujang', '200 g mixed seafood, chopped and patted dry', '2 tablespoons gochujang']
World.alias reproduces the released LLMCuisine-22 aliasing for 100.0% of 1,653 lines


## Build the cuisine and score it

`cuisine_from_lines` aliases each recipe's lines and keeps every recipe with at least one matched ingredient. The first `scorecard` call computes the reference statistics of the 22 real cuisines and caches them. These are the real Z of each cuisine (100,000-recipe null) and its category fingerprint, and the step takes about a minute.

As a control, we score a random sample of 160 genuine Korean recipes from CulinaryDB the same way. Its novelty is leave-one-out (`self_index`): each real recipe is compared with every *other* real recipe.

In [5]:
mine = world.cuisine_from_lines(f"{MODEL}:{REGION}", batch["ingredients"].tolist())
print(
    f"{len(mine)} of {len(batch)} recipes have at least one matched ingredient "
    f"({mine.sizes.mean():.1f} matched ingredients per recipe on average)"
)

t0 = time.perf_counter()
card = world.scorecard(mine, REGION)
print(f"scorecard in {time.perf_counter() - t0:.0f} s, including the reference statistics")
sub, positions = real_subsample(world.reference(), REGION, n=len(mine), rng=np.random.default_rng(0))
control = world.scorecard(sub, REGION, self_index=positions)

160 of 160 recipes have at least one matched ingredient (8.8 matched ingredients per recipe on average)


scorecard in 44 s, including the reference statistics


In [6]:
MEANING = {
    "n_recipes": "recipes scored (at least one matched ingredient)",
    "n_pairing_recipes": "recipes with ≥ 2 matched ingredients (Ns is defined for these)",
    "mean_size": "matched ingredients per recipe",
    "ns": "mean Ns: shared flavor compounds per ingredient pair, averaged over recipes",
    "ns_null": "mean Ns of 20,000 random recipes from the candidate's own ingredients and recipe sizes",
    "delta_ns": "ΔNs = ns − ns_null (> 0 uniform, < 0 contrasting food pairing)",
    "delta_ns_lo": "lower end of the 95% bootstrap CI of ΔNs (1,000 resamples of recipes, null mean fixed)",
    "delta_ns_hi": "upper end of that CI",
    "d": "effect size ΔNs / σ_null",
    "d_lo": "95% CI of d, lower end",
    "d_hi": "95% CI of d, upper end",
    "z_sb": "Singh & Bagler Z = √100,000 · d",
    "z_real": "Z of the full real cuisine (random null)",
    "sign_match": "z_sb has the same sign as z_real",
    "sign_error": "pre-registered rule: the ΔNs CI excludes 0 on the side opposite to z_real",
    "rel_abs_dz": "|z_sb − z_real| / |z_real|",
    "fingerprint_spearman": "Spearman ρ between the candidate's and the real category fingerprints (Z_cat)",
    "fingerprint_cosine": "cosine similarity of the same two Z_cat vectors",
    "fingerprint_n": "categories compared (each in ≥ 10 candidate recipes)",
    "prevalence_jsd": "Jensen-Shannon divergence of ingredient usage vs real (0 = same, 1 = disjoint)",
    "category_jsd": "JSD of the mean category composition of recipes vs real",
    "sai": "Stereotype Amplification Index on the real top-10 signature ingredients (1 = used as in real)",
    "freq_rank_mae": "MAE between normalized ingredient frequency-rank curves (Tuwani et al. 2019)",
    "size_ks": "Kolmogorov-Smirnov statistic of recipe sizes vs real",
    "size_wasserstein": "Wasserstein distance of recipe sizes vs real (in ingredients)",
    "unique_ingredients": "distinct ingredients used",
    "diversity_ratio": "unique ingredients / the real cuisine's at the same number of recipes (rarefied)",
    "novelty_mean": "mean over recipes of the maximum Jaccard similarity to any CulinaryDB recipe",
    "near_copy_rate": "share of recipes whose maximum Jaccard similarity is ≥ 0.8",
}
table = pd.DataFrame({f"{MODEL} ({len(mine)})": card, f"real sample ({len(sub)})": control}).drop(index="region")
table["meaning"] = [MEANING[k] for k in table.index]
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(table.map(lambda v: f"{v:.3f}" if isinstance(v, float) else v))

,openai-gpt-6-luna (160),real sample (160),meaning
n_recipes,160,160,recipes scored (at least one matched ingredient)
n_pairing_recipes,160,157,recipes with ≥ 2 matched ingredients (Ns is defined for these)
mean_size,8.806,9.525,matched ingredients per recipe
ns,39.050,40.532,"mean Ns: shared flavor compounds per ingredient pair, averaged over recipes"
ns_null,42.443,42.533,"mean Ns of 20,000 random recipes from the candidate's own ingredients and recipe sizes"
delta_ns,-3.393,-2.001,"ΔNs = ns − ns_null (> 0 uniform, < 0 contrasting food pairing)"
delta_ns_lo,-5.603,-4.637,"lower end of the 95% bootstrap CI of ΔNs (1,000 resamples of recipes, null mean fixed)"
delta_ns_hi,-1.010,0.981,upper end of that CI
d,-0.167,-0.091,effect size ΔNs / σ_null
d_lo,-0.276,-0.210,"95% CI of d, lower end"


**Reading the scorecard.**
- **Compare with the real-sample column, not with zero.** A genuine sample of a cuisine also differs from the full cuisine. The right yardstick is the spread over many equal-N samples. This notebook draws one; the D2 analysis draws 1,000 per region (`analysis/audit/real_subsamples.py`).
- **Z of a small sample.** `z_sb` uses the candidate's *own* ingredient pool. A small sample loses the cuisine's rare, low-sharing ingredients, and that shifts its null. The D2 analysis therefore judges food-pairing signs at equal N (the equal-N amendment in the plan's decision log). Also, Z grows with $N_{rand}$ and not with the number of recipes, so read `d` and the ΔNs CI together with it.
- **Fingerprint scores** rank only `fingerprint_n` categories. With few categories they are noisy.

## The same audit from the command line

`flavoraudit audit` takes a long-format CSV with one row per ingredient line and the columns `recipe_id`, `cuisine` (a CulinaryDB region name) and `line`. It writes one scorecard row for every claimed cuisine with at least 20 recipes; smaller cuisines are skipped.

The data folder can be given in two ways:
- the `FLAVORAUDIT_DATA` environment variable;
- `--data-dir`, a top-level option that must come *before* `audit` (`flavoraudit --data-dir data/raw audit recipes.csv`).

Every run rebuilds the reference statistics (about a minute).

In [7]:
workdir = Path(tempfile.mkdtemp(prefix="flavoraudit_cli_"))
picks = [("Korea", 25), ("Japan", 25), ("Scandinavia", 5)]  # 5 Scandinavian recipes: fewer than 20, so skipped
csv = pd.concat([llm[(llm["model"] == MODEL) & (llm["cuisine"] == c)].head(n) for c, n in picks])
csv = csv[["recipe_id", "cuisine", "ingredients"]].explode("ingredients").rename(columns={"ingredients": "line"})
csv.to_csv(workdir / "recipes.csv", index=False)
print(f"recipes.csv: {len(csv)} lines from {csv['recipe_id'].nunique()} recipes")
csv.head(4)

recipes.csv: 493 lines from 55 recipes


,recipe_id,cuisine,line
6899,f875b3871e0e1d050-0,Korea,2 cups short-grain rice
6899,f875b3871e0e1d050-0,Korea,"200 g beef, thinly sliced"
6899,f875b3871e0e1d050-0,Korea,1 tablespoon soy sauce
6899,f875b3871e0e1d050-0,Korea,1 teaspoon sesame oil


In [8]:
args = ["audit", "recipes.csv", "--out", "audit_out"]
print("$ FLAVORAUDIT_DATA=data/raw flavoraudit " + " ".join(args))
t0 = time.perf_counter()
run = subprocess.run(
    [sys.executable, "-m", "flavoraudit.cli", *args],
    cwd=workdir,
    env={**os.environ, "FLAVORAUDIT_DATA": str(DATA)},
    capture_output=True,
    text=True,
)
print(run.stderr + run.stdout + f"(exit code {run.returncode}, {time.perf_counter() - t0:.0f} s)")
assert run.returncode == 0

$ FLAVORAUDIT_DATA=data/raw flavoraudit audit recipes.csv --out audit_out


Japan: 25 recipes scored
Korea: 25 recipes scored
skipping Scandinavia: 5 recipes (< 20)
matched 89.0% of lines -> audit_out/scorecard.csv
(exit code 0, 55 s)


In [9]:
scores = pd.read_csv(workdir / "audit_out" / "scorecard.csv")
columns = ["region", "n_recipes", "z_sb", "z_real", "sign_error", "prevalence_jsd", "sai", "near_copy_rate"]
display(scores[columns].round(3))
pd.read_csv(workdir / "audit_out" / "aliased_lines.csv").head(5)  # entity_id is empty for unmatched lines

,region,n_recipes,z_sb,z_real,sign_error,prevalence_jsd,sai,near_copy_rate
0,Japan,25,-255.960,-125.062,False,0.391,0.820,0.20
1,Korea,25,-88.266,-55.368,False,0.178,1.084,0.28


,recipe_id,cuisine,line,entity_id
0,f875b3871e0e1d050-0,Korea,2 cups short-grain rice,55.0
1,f875b3871e0e1d050-0,Korea,"200 g beef, thinly sliced",270.0
2,f875b3871e0e1d050-0,Korea,1 tablespoon soy sauce,291.0
3,f875b3871e0e1d050-0,Korea,1 teaspoon sesame oil,299.0
4,f875b3871e0e1d050-0,Korea,1 teaspoon sugar,781.0


In [10]:
shutil.rmtree(workdir)  # remove the temporary CSV and the command-line outputs

Next: `03_copy_mutate_and_fingerprints.ipynb` builds the evolutionary baseline that the D2 audit compares LLM cuisines with.